# 01 · Customer fleet ownership

This notebook replicates the Customer fleet ownership paragraph of the report's Results section.

> Customer fleet ownership: Ticket serials reach 3,041 of 7,183 devices (42%), and accounts are individual facilities (845 accounts at 725–837 sites), so fleets come from the installed-base list.

**How it works.** We count the clearView devices that ticket serials reach in the per-device table from the telemetry database, and we count the accounts and their sites in the tables that `customer_fleet_audit.py` builds from IPA's case export. With `RERUN = True`, the notebook re-runs that script in a scratch copy of the outputs and compares each file it writes with the committed one; with `RERUN = False`, it reads the committed outputs. Either way, it recomputes every number from the tables in `OUT` and checks it against the report.

**Scripts and inputs.** The notebook re-runs `verification/customer_fleet_audit.py`, which writes the account and site tables (`fleet_*.csv` at the top of `verification/outputs/`). We use the telemetry-database outputs as committed inputs without re-running them, because the scripts that wrote them need the clearView telemetry database, which the notebooks do not re-read: `fu/fu_devices.csv` and `fu/fu_ticket_last10.csv` from `db_followup.py`, and `db/db_devices_summary.csv` and `db/db_linkage.csv` from `verify_with_db.py`. `docs/03_customer_fleet_ownership.md` (§3.1 and §3.4) describes both analyses.

In [1]:
RERUN = True  # False: recompute from the committed outputs without re-running the scripts
import pandas as pd
import replicate as R
OUT = R.outputs(RERUN)

RERUN is True: the scripts write into a scratch copy of verification/outputs/.


In [2]:
if RERUN:
    R.rerun(OUT, 'customer_fleet_audit.py')
R.compare(OUT)

customer_fleet_audit.py: exit status 0, 2 s
    done
15 files written: 8 DIFFERENT, 7 identical.


,file written by the re-run,compared with the committed file
0,fleet_account_granularity.csv,DIFFERENT (5 vs. 11 rows; columns same)
1,fleet_coverage_bounds.csv,identical
2,fleet_customer_naive_rates.csv,DIFFERENT (columns in_name_tier_multi_account_...
3,fleet_db_checks_spec.csv,identical
4,fleet_facility_vs_system.csv,DIFFERENT (column value)
5,fleet_host_expectations.csv,DIFFERENT (6 vs. 13 rows; columns same)
6,fleet_model_type_note.csv,identical
7,fleet_multi_account_serial_classes.csv,"DIFFERENT (columns relationship, rule_class, s..."
8,fleet_multi_account_serials.csv,"DIFFERENT (columns relationship, rule_class, s..."
9,fleet_naive_rate_summary.csv,identical


**About the comparison.** Besides IPA's case export, `customer_fleet_audit.py` reads a hand-made review, kept in the private folder, that groups the accounts into health systems, and the committed files were made with it. Where that file is missing, as in the run saved here, the script falls back to its name-token grouping, and the files that carry the reviewed grouping are listed as DIFFERENT because their reviewed-system rows or columns change: `fleet_account_granularity.csv`, `fleet_system_size_distribution.csv`, `fleet_shared_addresses.csv`, `fleet_customer_naive_rates.csv`, `fleet_multi_account_serials.csv`, `fleet_multi_account_serial_classes.csv`, `fleet_facility_vs_system.csv`, and `fleet_host_expectations.csv`. Without the review, the script does not write `fleet_segments.csv`, `fleet_review_evidence.csv`, or `fleet_l2b_cluster_review.csv`, so their committed copies stay in the scratch folder. The account and site counts do not depend on the review, so the parts of these files that this notebook reads (the accounts and their tickets, the accounts at shared addresses, and the account and site rows of the granularity table) are the same in the re-created and committed files.

## Devices that ticket serials reach

A device is a row of the clearView device table, and a ticket serial reaches it when the device ID equals the Serial # on at least one ticket, planned or unplanned, as exact text after trimming spaces. We count the devices in `fu/fu_devices.csv` (one row per device, from `db_followup.py`), and a device counts as reached when it has a ticket account, which the script fills only when the device ID is the serial on a ticket. We cross-check the count with the device IDs on the device-matched tickets in `fu/fu_ticket_last10.csv` and with `verify_with_db.py`'s count of device-table rows and its run of the midterm analysis's linkage query (`db/db_devices_summary.csv` and `db/db_linkage.csv`).

In [3]:
devices = R.read_csv(OUT, 'fu/fu_devices.csv')
assert devices.machine_id.is_unique
n_devices = len(devices)
reached = devices.ticket_customer.notna()
n_reached = int(reached.sum())
share_reached = n_reached / n_devices

# Cross-checks: device IDs on the device-matched tickets, and the linkage query as verify_with_db.py ran it
ticket_ids = set(R.read_csv(OUT, 'fu/fu_ticket_last10.csv', usecols=['machine_id']).machine_id)
assert ticket_ids == set(devices.machine_id[reached])
device_summary = R.read_csv(OUT, 'db/db_devices_summary.csv').iloc[0]
linkage = R.read_csv(OUT, 'db/db_linkage.csv').iloc[0]

print(f'Ticket serials reach {n_reached:,} of {n_devices:,} devices ({share_reached:.1%}).')
pd.DataFrame([
    ('fu/fu_devices.csv', 'devices, and those with a ticket account', R.num(n_devices), R.num(n_reached)),
    ('fu/fu_ticket_last10.csv', 'device IDs on the device-matched tickets', '',
     R.num(len(ticket_ids & set(devices.machine_id)))),
    ('db/db_devices_summary.csv, db/db_linkage.csv', 'device-table rows, and the linkage query',
     R.num(int(device_summary['rows'])), R.num(int(linkage['matched_machines']))),
], columns=['table', 'what it counts', 'devices', 'reached by a ticket serial'])

Ticket serials reach 3,041 of 7,183 devices (42.3%).


,table,what it counts,devices,reached by a ticket serial
0,fu/fu_devices.csv,"devices, and those with a ticket account","7,183","3,041"
1,fu/fu_ticket_last10.csv,device IDs on the device-matched tickets,,"3,041"
2,"db/db_devices_summary.csv, db/db_linkage.csv","device-table rows, and the linkage query","7,183","3,041"


## Accounts

An account is a ticket's Account Name after trimming spaces, which the midterm analysis calls a customer. We count the rows of `fleet_customer_naive_rates.csv`, which `customer_fleet_audit.py` writes with one row per account and its tickets.

In [4]:
accounts = R.read_csv(OUT, 'fleet_customer_naive_rates.csv')
assert accounts.customer.is_unique
n_accounts = len(accounts)
print(f'{n_accounts:,} accounts, which carry {accounts.tickets.sum():,} tickets.')

845 accounts, which carry 7,471 tickets.


## Sites: distinct address strings

Each account has one address string, "City,State ZIP+4" (or a Canadian postal code) with no street line, and `customer_fleet_audit.py` stops if an account has more than one. The distinct address strings therefore number the accounts minus the extra accounts at shared addresses, which we take from `fleet_shared_addresses.csv` (one row per address string that more than one account shares, with its accounts as pseudonyms).

In [5]:
shared = R.read_csv(OUT, 'fleet_shared_addresses.csv')
shared['accounts at the address'] = shared.customers.str.split(' + ', regex=False).str.len()
extra_accounts = int((shared['accounts at the address'] - 1).sum())
n_addresses = n_accounts - extra_accounts
print(f'{n_accounts:,} accounts minus {extra_accounts} extra accounts at the {len(shared)} shared address strings '
      f'leaves {n_addresses:,} distinct address strings.')
shared.groupby(['accounts at the address', 'address_precision']).size().rename('shared address strings').reset_index()

845 accounts minus 8 extra accounts at the 8 shared address strings leaves 837 distinct address strings.


,accounts at the address,address_precision,shared address strings
0,2,ZIP+4/postal code,7
1,2,ZIP5 only,1


## Sites: distinct City, State, and ZIP5

The lower end of the range is the number of distinct city, state, and 5-digit ZIP combinations, a lower bound on sites because it merges different organizations that share a ZIP5. No committed table lists the accounts' addresses, which are private, so we read this count from `fleet_account_granularity.csv`, where `customer_fleet_audit.py` counts it over the same accounts. We show the table's account and address-string counts next to our recounts above.

In [6]:
granularity = R.read_csv(OUT, 'fleet_account_granularity.csv')


def granularity_count(rule):
    """The count in the one granularity row whose rule starts with `rule`."""
    row = granularity[granularity.rule.str.startswith(rule)]
    assert len(row) == 1, rule
    return int(row['count'].iloc[0])


n_zip5_sites = granularity_count('distinct City+State+ZIP5')
print(f'{n_zip5_sites:,} distinct City + State + ZIP5 combinations among the {n_accounts:,} accounts.')
pd.DataFrame({'fleet_account_granularity.csv': [granularity_count('distinct cleaned Account Name'),
                                                granularity_count('distinct address string')],
              'recount above': [n_accounts, n_addresses]},
             index=['accounts', 'distinct address strings'])

725 distinct City + State + ZIP5 combinations among the 845 accounts.


,fleet_account_granularity.csv,recount above
accounts,845,845
distinct address strings,837,837


## Check against the report

In [7]:
R.check([
    ('Devices that ticket serials reach', '3,041', R.num(n_reached)),
    ('Devices in the telemetry device table', '7,183', R.num(n_devices)),
    ('Share of devices reached', '42%', R.pct(share_reached)),
    ('Accounts', '845', R.num(n_accounts)),
    ('Sites, ZIP5 combinations to address strings', '725–837', R.span(n_zip5_sites, n_addresses)),
]);

,Finding,Report,Recomputed,Match
0,Devices that ticket serials reach,"3,041","3,041",yes
1,Devices in the telemetry device table,"7,183","7,183",yes
2,Share of devices reached,42%,42%,yes
3,Accounts,845,845,yes
4,"Sites, ZIP5 combinations to address strings",725–837,725–837,yes


All 5 values match the report.
